# 15 — RAG Evaluation on WSU Domain Test Cases
Evaluates the full RAG pipeline on 120 WSU domain test cases.

Metrics: accuracy, retrieval precision, end-to-end latency, token usage.
Side-by-side comparison: RAG vs non-RAG accuracy by category.
Results saved to `data/results/rag_eval_results.json`.

In [1]:
import sys, os, json, time
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT = Path("../..")
sys.path.insert(0, str(REPO_ROOT / "src"))
load_dotenv(REPO_ROOT / ".env")

from retrieval.retriever import CourseRetriever
from retrieval.context_builder import ContextBuilder
from llm.claude_client import ClaudeClient
from llm.cache import ResponseCache
from evaluation.metrics import EvaluationMetrics

INDEX_DIR    = REPO_ROOT / "data" / "domain"
RESULTS_DIR  = REPO_ROOT / "data" / "results"
TEST_CASES   = REPO_ROOT / "data" / "domain" / "test_cases.json"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

retriever = CourseRetriever(index_dir=str(INDEX_DIR))
builder   = ContextBuilder(retriever, top_k=5)
cache     = ResponseCache(db_path=str(REPO_ROOT / "data" / "cache" / "responses.db"))
client    = ClaudeClient(model="claude-haiku-4-5", api_key=os.getenv("ANTHROPIC_API_KEY"))

print("Components loaded.")

/Users/jaime/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/jaime/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Components loaded.


## Load Test Cases

In [2]:
if not TEST_CASES.exists():
    print(f"WARNING: {TEST_CASES} not found.")
    test_cases = []
else:
    raw = json.loads(TEST_CASES.read_text())
    test_cases = (raw["cases"] if isinstance(raw, dict) else raw)[:120]
    print(f"Loaded {len(test_cases)} test cases.")
    categories = list({tc.get('category', 'general') for tc in test_cases})
    print(f"Categories: {categories}")

Loaded 120 test cases.
Categories: ['prerequisite_validation', 'credit_calculations', 'degree_progress', 'schedule_feasibility', 'ucore_planning', 'prerequisite_chain_discovery']


## Evaluation Runner

In [3]:
def call_llm(prompt, temperature=0.0):
    cached = cache.get(prompt, model=client.model, temperature=temperature)
    if cached:
        return cached, {"input_tokens": 0, "output_tokens": 0}
    resp = client.generate(prompt, temperature=temperature, max_tokens=400)
    cache.set(prompt, model=client.model, temperature=temperature, response=resp)
    stats = client.get_usage_stats()
    return resp, {"input_tokens": stats["total_input_tokens"], "output_tokens": stats["total_output_tokens"]}

def evaluate_rag(cases, use_rag=True):
    predictions, ground_truth = [], []
    latencies, token_counts = [], []
    retrieval_precisions = []

    for tc in cases:
        question = tc["question"]
        expected = tc["expected_answer"]
        relevant = set(tc.get("relevant_courses", []))

        t0 = time.time()
        if use_rag:
            prompt, sources = builder.build(question)
            retrieved_codes = {s["course_code"] for s in sources}
            if relevant:
                precision = len(retrieved_codes & relevant) / len(retrieved_codes) if retrieved_codes else 0
                retrieval_precisions.append(precision)
        else:
            prompt = f"You are a WSU academic advisor.\n\nQuestion: {question}"
            sources = []

        resp, usage = call_llm(prompt)
        latencies.append(time.time() - t0)
        token_counts.append(usage["input_tokens"] + usage["output_tokens"])
        predictions.append(resp)
        ground_truth.append(expected)

    acc = EvaluationMetrics.accuracy(predictions, ground_truth)
    avg_latency = sum(latencies) / max(len(latencies), 1)
    avg_tokens  = sum(token_counts) / max(len(token_counts), 1)
    avg_precision = sum(retrieval_precisions) / max(len(retrieval_precisions), 1) if retrieval_precisions else None

    return {
        "accuracy": round(acc, 4),
        "avg_latency_s": round(avg_latency, 3),
        "avg_tokens": round(avg_tokens, 1),
        "avg_retrieval_precision": round(avg_precision, 4) if avg_precision is not None else "n/a",
        "predictions": predictions,
        "ground_truth": ground_truth,
    }

## Run RAG vs Non-RAG

In [4]:
if not test_cases:
    print("No test cases loaded — skipping evaluation.")
else:
    print("Running RAG evaluation...")
    rag_results    = evaluate_rag(test_cases, use_rag=True)
    print("Running non-RAG evaluation...")
    no_rag_results = evaluate_rag(test_cases, use_rag=False)

    print(f"\nRAG     accuracy: {rag_results['accuracy']:.2%}  "
          f"latency: {rag_results['avg_latency_s']}s  "
          f"tokens: {rag_results['avg_tokens']}  "
          f"retrieval precision: {rag_results['avg_retrieval_precision']}")
    print(f"Non-RAG accuracy: {no_rag_results['accuracy']:.2%}  "
          f"latency: {no_rag_results['avg_latency_s']}s  "
          f"tokens: {no_rag_results['avg_tokens']}")

Running RAG evaluation...
Running non-RAG evaluation...

RAG     accuracy: 0.00%  latency: 0.021s  tokens: 0.0  retrieval precision: n/a
Non-RAG accuracy: 0.00%  latency: 0.008s  tokens: 0.0


## Side-by-Side by Category

In [5]:
if test_cases:
    from collections import defaultdict

    cat_map = defaultdict(list)
    for i, tc in enumerate(test_cases):
        cat_map[tc.get("category", "general")].append(i)

    print(f"{'Category':<25} {'RAG Acc':>9} {'No-RAG Acc':>11} {'Delta':>8}")
    print("-" * 58)
    category_rows = []
    for cat, idxs in sorted(cat_map.items()):
        rag_preds    = [rag_results['predictions'][i] for i in idxs]
        no_rag_preds = [no_rag_results['predictions'][i] for i in idxs]
        gt           = [rag_results['ground_truth'][i] for i in idxs]
        rag_acc    = EvaluationMetrics.accuracy(rag_preds, gt)
        no_rag_acc = EvaluationMetrics.accuracy(no_rag_preds, gt)
        delta = rag_acc - no_rag_acc
        print(f"{cat:<25} {rag_acc:>9.2%} {no_rag_acc:>11.2%} {delta:>+8.2%}")
        category_rows.append({"category": cat, "rag_accuracy": round(rag_acc, 4),
                               "no_rag_accuracy": round(no_rag_acc, 4), "delta": round(delta, 4)})

Category                    RAG Acc  No-RAG Acc    Delta
----------------------------------------------------------
credit_calculations           0.00%       0.00%   +0.00%
degree_progress               0.00%       0.00%   +0.00%
prerequisite_chain_discovery     0.00%       0.00%   +0.00%
prerequisite_validation       0.00%       0.00%   +0.00%
schedule_feasibility          0.00%       0.00%   +0.00%
ucore_planning                0.00%       0.00%   +0.00%


## Save Results

In [6]:
if test_cases:
    output = {
        "rag": {
            "accuracy":                rag_results["accuracy"],
            "avg_latency_s":           rag_results["avg_latency_s"],
            "avg_tokens":              rag_results["avg_tokens"],
            "avg_retrieval_precision": rag_results["avg_retrieval_precision"],
        },
        "no_rag": {
            "accuracy":    no_rag_results["accuracy"],
            "avg_latency_s": no_rag_results["avg_latency_s"],
            "avg_tokens":  no_rag_results["avg_tokens"],
        },
        "by_category": category_rows,
    }
    out_path = RESULTS_DIR / "rag_eval_results.json"
    out_path.write_text(json.dumps(output, indent=2))
    print(f"Results saved to {out_path}")

Results saved to ../../data/results/rag_eval_results.json
